# Maze predictive representations

In [ ]:
import json
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

In [ ]:

def format_p(p):
    s = f"{float(p):.3f}".rstrip("0")
    return s if not s.endswith(".") else s + "0"


def make_run_specs():
    runs = []

    for ds in RUN_SETS:
        for p in ds["ps"]:
            for seed in ds["seeds"]:
                run_dir = ds["root"] / ds["template"].format(
                    prefix=ds["prefix"],
                    p_text=format_p(p),
                    seed=seed,
                )

                runs.append({
                    "name": ds["name"],
                    "p": p,
                    "seed": str(seed),
                    "run_dir": run_dir,
                    "reward_node": ds["reward_node"],
                    "topology_file": ds["topology_file"],
                    "expected_bouts": ds["expected_bouts"],
                })

    return runs


def list_bout_paths(run_dir):
    matches = []

    for path in Path(run_dir).glob("bout_*_metrics.npz"):
        match = re.fullmatch(r"bout_(\d+)_metrics\.npz", path.name)
        if match:
            matches.append((int(match.group(1)), path))

    return sorted(matches, key=lambda x: x[0])


def inspect_one_bout(spec=None):
    specs = make_run_specs() if spec is None else [spec]

    for s in specs:
        paths = list_bout_paths(s["run_dir"])
        if not paths:
            continue

        bout_id, path = paths[0]

        with np.load(path, allow_pickle=True) as data:
            print(f"Run: {s['name']}, p={s['p']}, seed={s['seed']}")
            print(f"File: {path}, bout={bout_id}, files={len(paths)}")

            for key in data.files:
                print(key, data[key].shape, data[key].dtype)

            obs = np.asarray(data[OBS_KEY]).reshape(-1)
            print("Observations:", obs[:12])
            print("Transitions:", len(obs) - 1)

        return path

    print("No bout files found.")
    return None



## Maze topology

The symmetric maze is a 127-node binary tree. Maze depth is measured from the root, and transition direction is determined by the change in depth.


In [ ]:

@dataclass
class MazeTopology:
    neighbors: tuple
    depth: np.ndarray

    @property
    def n_nodes(self):
        return len(self.neighbors)

    @classmethod
    def symmetric(cls, n_nodes=127):
        adj = [set() for _ in range(n_nodes)]

        for child in range(1, n_nodes):
            parent = (child - 1) // 2
            adj[parent].add(child)
            adj[child].add(parent)

        return cls._from_neighbors(adj)

    @classmethod
    def from_matrix(cls, adjacency):
        mat = np.asarray(adjacency)

        if mat.ndim != 2 or mat.shape[0] != mat.shape[1]:
            raise ValueError("Topology adjacency must be a square matrix")
        if not np.all(np.isfinite(mat)):
            raise ValueError("Adjacency contains nonfinite values")

        adj = [set() for _ in range(mat.shape[0])]

        for u, v in np.argwhere(mat != 0):
            if u != v:
                adj[int(u)].add(int(v))
                adj[int(v)].add(int(u))

        return cls._from_neighbors(adj)

    @classmethod
    def _from_neighbors(cls, adj):
        n = len(adj)

        if n < 2:
            raise ValueError("Maze must have at least two nodes")

        edges = sum(len(v) for v in adj) // 2
        if edges != n - 1:
            raise ValueError(f"Expected {n - 1} edges, got {edges}")

        depth = np.full(n, -1, dtype=int)
        depth[0] = 0
        queue = [0]

        for u in queue:
            for v in sorted(adj[u]):
                if depth[v] < 0:
                    depth[v] = depth[u] + 1
                    queue.append(v)

        if np.any(depth < 0):
            raise ValueError("Topology is not connected to root 0")

        return cls(
            tuple(frozenset(x) for x in adj),
            depth,
        )

    def step_info(self, current, target, *, strict=True):
        current = np.asarray(current, dtype=np.int64).reshape(-1)
        target = np.asarray(target, dtype=np.int64).reshape(-1)

        if len(current) != len(target):
            raise ValueError("Source-target length mismatch")

        valid = np.array([
            0 <= u < self.n_nodes
            and 0 <= v < self.n_nodes
            and v in self.neighbors[int(u)]
            for u, v in zip(current, target)
        ], dtype=bool)

        if strict and not np.all(valid):
            bad = np.flatnonzero(~valid)[:6]
            edges = [(int(current[i]), int(target[i])) for i in bad]
            raise ValueError(f"Invalid maze edges at {bad.tolist()}: {edges}")

        direction = np.zeros(len(current), dtype=np.int8)
        direction[valid] = np.sign(
            self.depth[target[valid]] - self.depth[current[valid]]
        )

        if np.any(valid & (direction == 0)):
            raise AssertionError("Valid tree transition has zero depth change")

        return valid, direction


def load_topology(topology_file):
    if topology_file is None:
        return MazeTopology.symmetric(127)

    with np.load(topology_file, allow_pickle=False) as data:
        if "adjacency" not in data.files:
            raise KeyError("Topology NPZ must contain adjacency")

        return MazeTopology.from_matrix(data["adjacency"])


## Load and validate bouts

In [ ]:

def _transition_vector_validator(data, key, n_transitions):
    arr = np.asarray(data[key])

    if arr.ndim != 1 or len(arr) != n_transitions:
        raise ValueError(
            f"{key}: expected ({n_transitions},), got {arr.shape}"
        )

    return arr


def _prior_matrix_validator(data, key, n_transitions):
    arr = np.asarray(data[key], dtype=float)

    if arr.ndim != 2 or arr.shape[0] != n_transitions:
        raise ValueError(
            f"{key}: expected ({n_transitions}, latent_dim), got {arr.shape}"
        )

    if not np.all(np.isfinite(arr)):
        raise ValueError(f"{key}: nonfinite latent coordinates")

    return arr


def load_validated_bout(path, topology):
    keys = [OBS_KEY, ACTION_KEY, PRIOR_KEY, CE_KEY]

    with np.load(path, allow_pickle=False) as archive:
        missing = set(keys) - set(archive.files)
        if missing:
            raise KeyError(f"{path}: missing {sorted(missing)}")

        raw = {key: archive[key] for key in keys}

    obs = np.asarray(raw[OBS_KEY])
    if obs.ndim != 1 or not np.issubdtype(obs.dtype, np.integer):
        raise ValueError(
            f"{path}: expected 1D integer observations, "
            f"got {obs.shape}/{obs.dtype}"
        )

    n_transitions = len(obs) - 1
    if n_transitions < 1:
        raise ValueError(f"{path}: no transitions")

    action = _transition_vector_validator(
        {ACTION_KEY: raw[ACTION_KEY][1:]},
        ACTION_KEY,
        n_transitions,
    )
    ce = _transition_vector_validator(
        raw, CE_KEY, n_transitions
    ).astype(float)
    z = _prior_matrix_validator(raw, PRIOR_KEY, n_transitions)

    valid, direction = topology.step_info(obs[:-1], obs[1:])

    finite_ce = np.isfinite(ce) & (ce >= 0)
    if not np.all(finite_ce[valid]):
        raise ValueError(f"{path}: invalid prior CE values")

    if not np.all(np.isin(action, [0, 1, 2])):
        raise ValueError(f"{path}: unexpected action codes")

    return {
        "obs": obs.astype(np.int64),
        "x": obs[:-1].astype(np.int64),
        "y": obs[1:].astype(np.int64),
        "action": action.astype(np.int64),
        "z": z,
        "ce": ce,
        "valid": valid,
        "direction": direction,
        "dest_depth": topology.depth[
            obs[1:].clip(0, topology.n_nodes - 1)
        ],
        "n_invalid_edges": int((~valid).sum()),
        "n_transitions": n_transitions,
    }


## Metrics

- **Linear depth decodability:** Ridge regression predicting maze depth from prior latent representations, evaluated using five-fold cross-validation grouped by bout. Reported as held-out \(R^2\).
- **Trajectory tortuosity:** Latent path length divided by endpoint distance for consecutive inward or outward segments containing at least four transitions.
- **Local direction consistency:** Fraction of five nearest neighboring latent transitions with the same direction. Repeated occurrences of each directed transition are averaged before computing neighbors.
- **Transition-balanced cross-entropy:** Mean prior prediction error across unique directed transitions, weighted equally.
- **Reward performance:** Success rate and number of transitions to the first reward encounter, with steps to reward averaged over successful bouts.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

In [ ]:
MIN_SEGMENT_TRANSITIONS = 4
K_NEIGHBORS = 5
MIN_LOCAL_DEST_DEPTH = 5
MIN_LOCAL_POINTS = 50

In [ ]:

def monotonic_segments(record, min_length=MIN_SEGMENT_TRANSITIONS):
    dirs = record["direction"]
    start = 0

    for t in range(1, len(dirs) + 1):
        if t == len(dirs) or dirs[t] != dirs[start]:
            if t - start >= min_length:
                yield np.arange(start, t)
            start = t


def linear_depth_alignment(records, direction="all", n_splits=5):
    if direction not in ("all", "outward", "inward"):
        raise ValueError("Invalid direction")

    z_parts = []
    depth_parts = []
    group_parts = []
    direction_parts = []

    for bout_id, record in enumerate(records):
        for idx in monotonic_segments(record):
            segment_dir = record["direction"][idx[0]]

            if direction == "outward" and segment_dir != 1:
                continue
            if direction == "inward" and segment_dir != -1:
                continue

            z_parts.append(record["z"][idx])
            depth_parts.append(record["dest_depth"][idx])
            group_parts.append(np.full(len(idx), bout_id))
            direction_parts.append(record["direction"][idx])

    if not z_parts:
        raise ValueError("No segments found")

    X = np.concatenate(z_parts)
    y = np.concatenate(depth_parts)
    groups = np.concatenate(group_parts)
    directions = np.concatenate(direction_parts)

    if len(np.unique(groups)) < n_splits:
        raise ValueError("Not enough bouts for cross-validation")

    cv = GroupKFold(n_splits=n_splits)

    predictions = np.full(len(y), np.nan)
    fold_r2 = []
    fold_mae = []

    for train_idx, test_idx in cv.split(X, y, groups):
        model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=1.0)
        )

        model.fit(X[train_idx], y[train_idx])
        pred = model.predict(X[test_idx])

        predictions[test_idx] = pred

        fold_r2.append(r2_score(y[test_idx], pred))
        fold_mae.append(mean_absolute_error(y[test_idx], pred))

    result = {
        "r2": r2_score(y, predictions),
        "mae": mean_absolute_error(y, predictions),
        "fold_r2_mean": np.mean(fold_r2),
        "fold_r2_std": np.std(fold_r2, ddof=1),
        "fold_mae_mean": np.mean(fold_mae),
        "fold_mae_std": np.std(fold_mae, ddof=1),
        "n_bouts": len(np.unique(groups)),
        "n_points": len(y),
    }

    if direction == "all":
        for name, sign in [("outward", 1), ("inward", -1)]:
            mask = directions == sign

            if mask.sum() >= 2 and np.ptp(y[mask]) > 0:
                result[f"{name}_r2"] = r2_score(
                    y[mask], predictions[mask]
                )
                result[f"{name}_mae"] = mean_absolute_error(
                    y[mask], predictions[mask]
                )

    return result


def trajectory_tortuosity(z):
    z = np.asarray(z, dtype=float)
    if z.ndim != 2 or len(z) < MIN_SEGMENT_TRANSITIONS: return np.nan
    steps = np.linalg.norm(np.diff(z, axis=0), axis=1)
    length = float(np.sum(steps))
    chord = float(np.linalg.norm(z[-1] - z[0]))
    if not np.isfinite(length + chord) or chord <= max(1e-12, 1e-10*length):
        return np.nan
    value = length / chord
    return value


def unique_transitions(records):
    pairs, latents, directions, depths = [], [], [], []

    for record in records:
        obs = record["obs"]
        valid = record["valid"]

        pairs.append(np.column_stack([
            obs[:-1][valid],
            obs[1:][valid]
        ]))

        latents.append(record["z"][valid])
        directions.append(record["direction"][valid])
        depths.append(record["dest_depth"][valid])

    pairs = np.concatenate(pairs)
    z = np.concatenate(latents)
    direction = np.concatenate(directions)
    depth = np.concatenate(depths)

    edges, first_idx, inverse, counts = np.unique(
        pairs, axis=0,
        return_index=True,
        return_inverse=True,
        return_counts=True
    )

    z_sum = np.zeros((len(edges), z.shape[1]))
    np.add.at(z_sum, inverse, z)

    z_unique = z_sum / counts[:, None]

    return (
        z_unique,
        direction[first_idx],
        depth[first_idx]
    )


def local_transition_consistency(z, direction, dest_depth,
                                 k=K_NEIGHBORS, min_depth=MIN_LOCAL_DEST_DEPTH,
                                 min_points=MIN_LOCAL_POINTS, return_points=False):
    z = np.asarray(z, dtype=float)
    direction = np.asarray(direction, dtype=np.int8)
    dest_depth = np.asarray(dest_depth, dtype=np.int64)

    if z.ndim != 2 or len(z) != len(direction) or len(z) != len(dest_depth):
        raise ValueError("kNN inputs must align one point per transition")

    mask = (dest_depth >= min_depth) & np.isin(direction, [-1, 1])
    Z, D = z[mask], direction[mask]
    n = len(Z)

    result = dict(consistency=np.nan, n_local_points=n,
                  outward_fraction=np.nan, chance_baseline=np.nan,
                  n_outward=int((D == 1).sum()), n_inward=int((D == -1).sum()),
                  effective_k=np.nan)

    if n:
        frac = float(np.mean(D == 1))
        result["outward_fraction"] = frac
        result["chance_baseline"] = ((result["n_outward"] * (result["n_outward"]-1)
                                     + result["n_inward"] * (result["n_inward"]-1))
                                     / (n * (n-1))) if n > 1 else np.nan

    if n < max(min_points, k+1) or len(np.unique(D)) < 2:
        return (result, np.array([], dtype=float)) if return_points else result

    if not np.all(np.isfinite(Z)):
        raise ValueError("Nonfinite kNN points")

    tree = cKDTree(Z)
    nearest = tree.query(Z, k=k+1, workers=1)[1]

    if nearest.ndim == 1: nearest = nearest[:, None]

    point_C = np.empty(n, dtype=float)
    nonself = nearest != np.arange(n)[:, None]
    select = nonself & (np.cumsum(nonself, axis=1) <= k)

    if select.sum() != n*k:
        raise AssertionError("Not enough nonself neighbors")

    nbr = nearest[select].reshape(n, k)
    point_C = (D[nbr] == D[:, None]).mean(axis=1)

    result["consistency"] = float(point_C.mean())
    result["effective_k"] = int(k)

    return (result, point_C) if return_points else result


def transition_balanced_ce(records, n_possible=252):
    edges_all = []
    ce_all = []

    for record in records:
        good = record["valid"] & np.isfinite(record["ce"])

        edges = np.column_stack([
            record["obs"][:-1][good],
            record["obs"][1:][good]
        ])

        edges_all.append(edges)
        ce_all.append(record["ce"][good])

    edges = np.concatenate(edges_all)
    ce = np.concatenate(ce_all)

    if len(ce) == 0:
        return dict(ce=np.nan, coverage=0.0, n_edges=0)

    unique_edges, inv = np.unique(
        edges, axis=0, return_inverse=True
    )

    totals = np.bincount(inv, weights=ce)
    counts = np.bincount(inv)
    mean_ce_per_edge = totals / counts

    return dict(
        ce=float(mean_ce_per_edge.mean()),
        coverage=len(unique_edges) / n_possible,
        n_edges=len(unique_edges)
    )


def reward_outcome(record, reward_node):
    idx = np.flatnonzero(record["obs"] == int(reward_node))
    horizon = int(record["n_transitions"])
    hit = int(len(idx) > 0)
    steps = int(idx[0]) if hit else None

    return dict(hit=hit, steps_success=float(steps) if hit else np.nan,
                steps_capped=float(steps if hit else horizon), horizon=horizon)


## Window analysis

Metrics are calculated separately for each run and training window. Results are saved at the window, bout, and segment levels.

Summary statistics are computed within each seed. Error bars across experiments are calculated from independent seeds.

In [ ]:

def select_window(paths, frac_start, frac_end):
    if not (0 <= frac_start < frac_end <= 1):
        raise ValueError('Window must satisfy 0 <= start < end <= 1')

    n = len(paths)
    start = int(round(frac_start * n))
    end = int(round(frac_end * n))
    return paths[start:end]


def analyze_window(spec, topology, paths, frac_start, frac_end):
    wpaths = select_window(paths, frac_start, frac_end)
    if not wpaths:
        raise ValueError(f'Empty window {frac_start}-{frac_end}')

    common = dict(
        dataset=spec['name'],
        p=spec['p'],
        seed=spec['seed'],
        window_start=frac_start,
        window_end=frac_end
    )

    records = []
    bout_rows = []
    segment_rows = []
    action_dynamics = {}
    invalid_edges = 0
    bouts_with_segments = set()

    # Load bouts
    for bid, path in wpaths:
        try:
            rec = load_validated_bout(path, topology)
        except Exception as e:
            raise RuntimeError(
                f'Failed run {spec["name"]}, p={spec["p"]}, '
                f'seed={spec["seed"]}, bout={bid}, path={path}'
            ) from e

        records.append(rec)
        invalid_edges += rec['n_invalid_edges']

        # Check action consistency
        for u, a, v, is_valid in zip(
            rec['obs'][:-1],
            rec['action'],
            rec['obs'][1:],
            rec['valid']
        ):
            if not is_valid:
                continue

            key = (int(u), int(a))
            v = int(v)

            if key in action_dynamics and action_dynamics[key] != v:
                raise ValueError(
                    f'Inconsistent (node, action) -> destination '
                    f'for {key}: {action_dynamics[key]} vs {v}. '
                    'Check action alignment and encoding.'
                )

            action_dynamics[key] = v

        reward = reward_outcome(rec, spec['reward_node'])

        bout_rows.append(dict(
            **common,
            bout_id=bid,
            **reward,
            n_invalid_edges=rec['n_invalid_edges']
        ))

        for seg_id, idx in enumerate(monotonic_segments(rec)):
            bouts_with_segments.add(bid)

            segment_rows.append(dict(
                **common,
                bout_id=bid,
                segment_id=seg_id,
                direction=(
                    'outward' if rec['direction'][idx[0]] == 1
                    else 'inward'
                ),
                n_steps=len(idx),
                tortuosity=trajectory_tortuosity(rec['z'][idx])
            ))

    # Prediction and geometry metrics
    ce_result = transition_balanced_ce(
        records,
        n_possible=2 * (topology.n_nodes - 1)
    )

    z_unique, direction_unique, depth_unique = unique_transitions(records)

    local = local_transition_consistency(
        z_unique,
        direction_unique,
        depth_unique,
        return_points=False
    )

    point_scores = None
    depth_result = {}

    if len(bouts_with_segments) >= 5:
        depth_result = linear_depth_alignment(
            records,
            direction='all',
            n_splits=5
        )

    df_bouts = pd.DataFrame(bout_rows)

    tort_valid = [
        r['tortuosity']
        for r in segment_rows
        if np.isfinite(r['tortuosity'])
    ]

    mean_tortuosity = (
        float(np.mean(tort_valid)) if tort_valid else np.nan
    )

    # Window summary
    result = dict(
        **common,

        n_bouts=len(records),
        n_transitions=int(df_bouts['horizon'].sum()),

        # Depth decodability
        linear_depth_r2=depth_result.get('r2', np.nan),
        linear_depth_mae=depth_result.get('mae', np.nan),

        linear_depth_r2_outward=depth_result.get(
            'outward_r2', np.nan
        ),
        linear_depth_mae_outward=depth_result.get(
            'outward_mae', np.nan
        ),
        linear_depth_r2_inward=depth_result.get(
            'inward_r2', np.nan
        ),
        linear_depth_mae_inward=depth_result.get(
            'inward_mae', np.nan
        ),

        linear_depth_fold_r2_mean=depth_result.get(
            'fold_r2_mean', np.nan
        ),
        linear_depth_fold_r2_std=depth_result.get(
            'fold_r2_std', np.nan
        ),
        linear_depth_fold_mae_mean=depth_result.get(
            'fold_mae_mean', np.nan
        ),
        linear_depth_fold_mae_std=depth_result.get(
            'fold_mae_std', np.nan
        ),

        # Geometry
        tortuosity=mean_tortuosity,
        transition_consistency=local['consistency'],

        # Prediction
        transition_balanced_ce=ce_result['ce'],
        n_unique_transitions=ce_result['n_edges'],
        transition_coverage=ce_result['coverage'],

        # Reward
        success_rate=float(df_bouts['hit'].mean()),
        steps_success=float(df_bouts['steps_success'].mean()),
        steps_capped=float(df_bouts['steps_capped'].mean()),

        **local
    )

    # Quality control
    qc = dict(
        **common,

        n_invalid_edges=invalid_edges,
        n_segments_total=len(segment_rows),
        n_segments_tort_valid=len(tort_valid),
        n_bouts_with_segments=len(bouts_with_segments),

        n_bouts_depth_cv=depth_result.get('n_bouts', 0),
        n_points_depth_cv=depth_result.get('n_points', 0),

        n_unique_transitions=ce_result['n_edges'],
        transition_coverage=ce_result['coverage'],

        n_bouts_hit=int(df_bouts['hit'].sum()),

        first_bout_id=int(wpaths[0][0]),
        last_bout_id=int(wpaths[-1][0])
    )

    return result, bout_rows, segment_rows, qc, point_scores


def analyze_run(spec, topology):
    paths = list_bout_paths(spec['run_dir'])

    if not paths:
        raise FileNotFoundError(f'No bout files in {spec["run_dir"]}')

    ids = [b for b, _ in paths]

    if len(set(ids)) != len(ids):
        raise ValueError('Duplicate bout IDs')

    if spec['expected_bouts'] is not None:
        if len(paths) != spec['expected_bouts']:
            raise ValueError(
                f'Expected {spec["expected_bouts"]} bout files in {spec["run_dir"]}; '
                f'found {len(paths)} (first IDs: {ids[:3]}, last IDs: {ids[-3:]})'
            )

        if ids != list(range(ids[0], ids[0] + len(ids))):
            raise ValueError(f'Bout IDs have gaps for {spec["run_dir"]}')

    summaries, bouts, segments, qc, point_outputs = [], [], [], [], {}

    for f0, f1 in WINDOWS:
        row, b, seg, q, point_C = analyze_window(
            spec, topology, paths, f0, f1
        )
        summaries.append(row)
        bouts.extend(b)
        segments.extend(seg)
        qc.append(q)

    return summaries, bouts, segments, qc, point_outputs


In [ ]:

METRIC_COLUMNS = [
    'linear_depth_r2',
    'linear_depth_mae',
    'linear_depth_r2_outward',
    'linear_depth_mae_outward',
    'linear_depth_r2_inward',
    'linear_depth_mae_inward',
    'linear_depth_fold_r2_mean',
    'linear_depth_fold_r2_std',
    'linear_depth_fold_mae_mean',
    'linear_depth_fold_mae_std',
    'tortuosity',
    'transition_consistency',
    'transition_balanced_ce',
    'n_unique_transitions',
    'transition_coverage',
    'success_rate',
    'steps_success',
    'steps_capped',
]


def save_numeric_outputs(summary_df, bouts_df, segments_df, qc_df, settings):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    summary_df.to_csv(OUTPUT_DIR / 'summary.csv', index=False)
    bouts_df.to_csv(OUTPUT_DIR / 'per_bout.csv', index=False)
    segments_df.to_csv(OUTPUT_DIR / 'per_segment.csv', index=False)
    qc_df.to_csv(OUTPUT_DIR / 'qc.csv', index=False)

    cols = [
        'dataset', 'p', 'seed', 'window_start', 'window_end',
        'n_bouts', 'n_transitions', *METRIC_COLUMNS,
        'n_local_points', 'n_outward', 'n_inward',
        'outward_fraction', 'chance_baseline'
    ]

    def write_npz_table(df, file):
        # Save string columns without pickle
        arrays = {}

        for col in df.columns:
            series = df[col]

            if series.dtype == object:
                arrays[col] = series.fillna('').astype(str).to_numpy(dtype=str)
            else:
                arrays[col] = series.to_numpy()

        np.savez_compressed(OUTPUT_DIR / file, **arrays)

    write_npz_table(summary_df[cols], 'summary_arrays.npz')
    write_npz_table(bouts_df, 'per_bout_arrays.npz')
    write_npz_table(segments_df, 'per_segment_arrays.npz')
    write_npz_table(qc_df, 'qc_arrays.npz')

    with open(OUTPUT_DIR / 'settings.json', 'w') as f:
        json.dump(settings, f, indent=2, default=str)

    print('SAVED:', OUTPUT_DIR.resolve())
    print('Files: four .npz arrays, four CSVs, and settings.json')


def execute_analysis(specs):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    available = [s for s in specs if list_bout_paths(s['run_dir'])]
    missing = [s for s in specs if not list_bout_paths(s['run_dir'])]

    print(f'Found {len(available)} runs, missing {len(missing)} / {len(specs)}.')

    if missing:
        pd.DataFrame([
            {k: v for k, v in s.items() if k != 'topology_file'}
            for s in missing
        ]).to_csv(OUTPUT_DIR / 'missing_runs.csv', index=False)

        print('Missing paths listed in:', OUTPUT_DIR / 'missing_runs.csv')

    if not available:
        print('No data yet at these paths. Set DATA_ROOT / RUN_SETS then rerun.')
        return None

    all_summary, all_bouts, all_segments, all_qc = [], [], [], []
    loaded_topos = {}

    for i, spec in enumerate(available, 1):
        topo_key = str(spec['topology_file'])

        if topo_key not in loaded_topos:
            loaded_topos[topo_key] = load_topology(spec['topology_file'])

        print(
            f'[{i}/{len(available)}] {spec["name"]} '
            f'p={spec["p"]} seed={spec["seed"]}',
            flush=True
        )

        a, b, c, d, point_outputs = analyze_run(
            spec, loaded_topos[topo_key]
        )

        all_summary.extend(a)
        all_bouts.extend(b)
        all_segments.extend(c)
        all_qc.extend(d)

    summary = pd.DataFrame(all_summary)
    bouts = pd.DataFrame(all_bouts)
    segments = pd.DataFrame(all_segments)
    qc = pd.DataFrame(all_qc)

    settings = dict(
        windows=WINDOWS,

        OBS_KEY=OBS_KEY,
        ACTION_KEY=ACTION_KEY,
        PRIOR_KEY=PRIOR_KEY,
        CE_KEY=CE_KEY,

        MIN_SEGMENT_TRANSITIONS=MIN_SEGMENT_TRANSITIONS,
        K_NEIGHBORS=K_NEIGHBORS,
        MIN_LOCAL_DEST_DEPTH=MIN_LOCAL_DEST_DEPTH,
        MIN_LOCAL_POINTS=MIN_LOCAL_POINTS,

        depth_metric='Ridge regression, 5-fold GroupKFold by bout',
        depth_target='destination node depth',
        depth_direction='shared decoder for inward and outward',
        depth_regularization_alpha=1.0,

        tortuosity_policy='per monotonic segment, averaged across window',

        transition_consistency_policy='one mean latent vector per unique directed edge',
        local_neighbor_space='original prior latent, Euclidean; self excluded',

        ce_policy='balanced over unique directed edges across entire window',

        segment_policy='maximal contiguous inward/outward within one bout',
        reward_failures='capped at horizon T, separate success_rate',

        run_sets=RUN_SETS,
        n_found=len(available),
        n_missing=len(missing)
    )

    save_numeric_outputs(summary, bouts, segments, qc, settings)

    return summary, bouts, segments, qc


In [ ]:

DATA_ROOT = Path("runs")
OUTPUT_DIR = Path("maze_metric_outputs_count")

RUN_SETS = [
    dict(
        name="Count",
        root=DATA_ROOT / "COUNT_REWARD_DRIVEN_1000",
        prefix="COUNT_REWARD_DRIVEN_1000_runs_slurm_20_window_agent_rew_p_",
        template="{prefix}{p_text}/{seed}",
        ps=[0.0, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05,
            0.06, 0.07, 0.08, 0.09, 0.1],
        seeds=list(range(15)),
        topology_file=None,
        reward_node=116,
        expected_bouts=1000,
    ),
]

WINDOWS = [
    (0.1, 0.2),
    (0.2, 0.3),
    (0.3, 0.4),
    (0.4, 0.5),
    (0.5, 0.6),
    (0.6, 0.7),
    (0.7, 0.8),
    (0.8, 0.9),
    (0.9, 1.0),
]

OBS_KEY = "obs_full"
ACTION_KEY = "act_full"
PRIOR_KEY = "z_p"
CE_KEY = "ce_p"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
specs = make_run_specs()

print(f"Runs: {len(specs)}")
print(f"Windows: {len(WINDOWS)}")

execute_analysis(specs)

In [ ]:
summary = pd.read_csv(OUTPUT_DIR / "summary.csv")
display(summary.head())

## Reward-driven fraction and node-visitation entropy

In [ ]:

ROOT = Path.cwd()

METRICS_DIR = ROOT / "maze_metric_outputs_random"
DATA_ROOT = ROOT / "runs"

RAW_ROOT = DATA_ROOT / "RANDOM_REWARD_DRIVEN_1000"
PREFIX = "RANDOM_REWARD_DRIVEN_1000_runs_slurm_20_window_agent_rew_p_"

BEHAVIOR_WINDOWS = [(0.2, 0.3), (0.9, 1.0)]
N_NODES = 127


def format_p(p):
    s = f"{float(p):.3f}".rstrip("0")
    return s if not s.endswith(".") else s + "0"


def get_bout_files(run_dir):
    files = []

    for path in run_dir.glob("bout_*_metrics.npz"):
        match = re.fullmatch(r"bout_(\d+)_metrics\.npz", path.name)
        if match:
            files.append((int(match.group(1)), path))

    return [path for _, path in sorted(files)]


def calculate_behavior(files):
    node_counts = np.zeros(N_NODES, dtype=np.int64)

    n_reward_actions = 0
    n_actions = 0

    for path in files:
        with np.load(path, allow_pickle=True) as data:
            obs = np.asarray(data["obs_full"], dtype=int)
            modes = np.asarray(data["action_mode_full"]).astype(str)

        if len(obs) != len(modes) or len(obs) < 2:
            raise ValueError(f"Invalid array alignment: {path}")

        if modes[0] != "dummy":
            raise ValueError(f"Expected dummy action mode: {path}")

        modes = modes[1:]
        destinations = obs[1:]

        if not np.all(np.isin(modes, ["reward_driven", "explore"])):
            raise ValueError(f"Unexpected action modes: {path}")

        if np.any((destinations < 0) | (destinations >= N_NODES)):
            raise ValueError(f"Invalid node ID: {path}")

        n_reward_actions += np.sum(modes == "reward_driven")
        n_actions += len(modes)

        node_counts += np.bincount(
            destinations, minlength=N_NODES
        )

    if n_actions == 0:
        raise ValueError("No transitions in selected window")

    reward_fraction = n_reward_actions / n_actions

    probabilities = node_counts[node_counts > 0] / node_counts.sum()
    node_entropy = -np.sum(probabilities * np.log(probabilities))

    return reward_fraction, node_entropy


summary = pd.read_csv(
    METRICS_DIR / "summary.csv",
    dtype={"seed": str}
)

conditions = summary[["p", "seed"]].drop_duplicates()

rows = []

for i, (p, seed) in enumerate(
    conditions.itertuples(index=False, name=None), 1
):
    run_dir = RAW_ROOT / f"{PREFIX}{format_p(p)}" / str(seed)
    files = get_bout_files(run_dir)

    if not files:
        raise FileNotFoundError(f"No bout files in {run_dir}")

    for w0, w1 in BEHAVIOR_WINDOWS:
        n = len(files)

        start = int(round(w0 * n))
        end = int(round(w1 * n))

        selected = files[start:end]
        fraction, entropy = calculate_behavior(selected)

        rows.append({
            "p": float(p),
            "seed": str(seed),
            "window_start": w0,
            "window_end": w1,
            "reward_driven_fraction": fraction,
            "node_entropy": entropy,
            "n_bouts": len(selected)
        })

    if i % 15 == 0 or i == len(conditions):
        print(f"Completed {i}/{len(conditions)} runs", flush=True)


behavior = pd.DataFrame(rows)

behavior.to_csv(
    METRICS_DIR / "behavior_metrics.csv",
    index=False
)

print("Saved behavior_metrics.csv")
display(behavior.head())


### Mice metrics

In [ ]:
MOUSE_DATA_DIR = (
    Path("runs")
    / "runs_20_window"
    / "water_sparsity_0.1_hdim_64_zdim_16_beta1e-3_lr1e-3_1e-3_1e-3"
)

BOUT_FRACTION = 0.2

OUTPUT_DIR = Path("maze_metric_outputs_mice_last0.2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MICE_TO_ANALYZE = None

OBS_KEY = "obs_full"
ACTION_KEY = "act_full"
PRIOR_KEY = "z_p"
CE_KEY = "ce_p"

MIN_SEGMENT_TRANSITIONS = 4
K_NEIGHBORS = 5
MIN_LOCAL_DEST_DEPTH = 5
MIN_LOCAL_POINTS = 50

print("Data:", MOUSE_DATA_DIR)
print("Output:", OUTPUT_DIR)

In [ ]:
def load_mouse_last_bouts(mouse, topology):
    run_dir = MOUSE_DATA_DIR / mouse
    files = list_bout_paths(run_dir)

    if not files:
        raise FileNotFoundError(f"No bout files found for {mouse}")

    n_total = len(files)
    n_selected = max(1, int(np.ceil(n_total * BOUT_FRACTION)))
    selected_files = files[-n_selected:]

    selected = []

    for bout_id, path in selected_files:
        rec = load_validated_bout(path, topology)
        selected.append((bout_id, rec, 0))

    n_steps = sum(
        rec["n_transitions"]
        for _, rec, _ in selected
    )

    print(
        f"{mouse}: {n_selected}/{n_total} bouts, "
        f"{n_steps} transitions"
    )

    return selected

In [ ]:

def calculate_mouse_metrics(mouse, selected, topology):
    records = [rec for _, rec, _ in selected]

    bout_rows = []
    segment_rows = []
    bouts_with_segments = set()

    for bout_id, rec, offset in selected:
        reward = reward_outcome(rec, reward_node=116)

        bout_rows.append({
            "mouse_id": mouse,
            "bout_id": bout_id,
            "first_selected_step": offset,
            **reward
        })

        for seg_id, idx in enumerate(monotonic_segments(rec)):
            bouts_with_segments.add(bout_id)

            tort = trajectory_tortuosity(rec["z"][idx])

            segment_rows.append({
                "mouse_id": mouse,
                "bout_id": bout_id,
                "segment_id": seg_id,
                "direction": (
                    "outward"
                    if rec["direction"][idx[0]] == 1
                    else "inward"
                ),
                "n_steps": len(idx),
                "tortuosity": tort,
            })

    depth_result = {}

    if len(bouts_with_segments) >= 5:
        depth_result = linear_depth_alignment(
            records,
            direction="all",
            n_splits=5
        )

    tort_values = [
        row["tortuosity"]
        for row in segment_rows
        if np.isfinite(row["tortuosity"])
    ]

    mean_tortuosity = (
        float(np.mean(tort_values))
        if tort_values else np.nan
    )

    z_unique, direction_unique, depth_unique = unique_transitions(records)

    local = local_transition_consistency(
        z_unique,
        direction_unique,
        depth_unique,
        k=K_NEIGHBORS,
        min_depth=MIN_LOCAL_DEST_DEPTH,
        min_points=MIN_LOCAL_POINTS
    )

    ce_result = transition_balanced_ce(
        records,
        n_possible=2 * (topology.n_nodes - 1)
    )

    df_bouts = pd.DataFrame(bout_rows)

    success_rate = float(df_bouts["hit"].mean())
    steps_success = float(df_bouts["steps_success"].mean())
    steps_capped = float(df_bouts["steps_capped"].mean())

    nodes = np.concatenate([
        rec["y"] for rec in records
    ])

    if np.any((nodes < 0) | (nodes >= topology.n_nodes)):
        raise ValueError("Invalid maze node IDs")

    counts = np.bincount(
        nodes,
        minlength=topology.n_nodes
    )

    probabilities = counts[counts > 0] / counts.sum()

    entropy = float(
        -np.sum(probabilities * np.log(probabilities))
    )

    entropy_normalized = entropy / np.log(topology.n_nodes)

    result = {
        "dataset": "Mouse",
        "seed": mouse,
        "mouse_id": mouse,

        "n_transitions": sum(
            rec["n_transitions"] for rec in records
        ),
        "n_bouts": len(records),

        "first_bout_id": selected[0][0],
        "last_bout_id": selected[-1][0],
        "first_bout_step_offset": selected[0][2],

        "linear_depth_r2": depth_result.get("r2", np.nan),
        "linear_depth_mae": depth_result.get("mae", np.nan),
        "linear_depth_r2_outward": depth_result.get(
            "outward_r2", np.nan
        ),
        "linear_depth_mae_outward": depth_result.get(
            "outward_mae", np.nan
        ),
        "linear_depth_r2_inward": depth_result.get(
            "inward_r2", np.nan
        ),
        "linear_depth_mae_inward": depth_result.get(
            "inward_mae", np.nan
        ),
        "linear_depth_fold_r2_mean": depth_result.get(
            "fold_r2_mean", np.nan
        ),
        "linear_depth_fold_r2_std": depth_result.get(
            "fold_r2_std", np.nan
        ),
        "linear_depth_fold_mae_mean": depth_result.get(
            "fold_mae_mean", np.nan
        ),
        "linear_depth_fold_mae_std": depth_result.get(
            "fold_mae_std", np.nan
        ),

        "tortuosity": mean_tortuosity,
        "transition_consistency": local["consistency"],

        "transition_balanced_ce": ce_result["ce"],
        "transition_coverage": ce_result["coverage"],
        "n_unique_transitions": ce_result["n_edges"],

        "success_rate": success_rate,
        "steps_success": steps_success,
        "steps_capped": steps_capped,

        "node_entropy": entropy,
        "node_entropy_normalized": entropy_normalized,

        **local
    }

    assert result["n_transitions"] == sum(
        rec["n_transitions"]
        for _, rec, _ in selected
    )

    return result, bout_rows, segment_rows


In [ ]:

if MICE_TO_ANALYZE is None:
    mice = sorted([
        d.name
        for d in MOUSE_DATA_DIR.iterdir()
        if d.is_dir() and list_bout_paths(d)
    ])
else:
    mice = MICE_TO_ANALYZE

print(f"Processing {len(mice)} mice")

topology = MazeTopology.symmetric(127)

all_summary = []
all_bouts = []
all_segments = []
skipped = []

for mouse in mice:
    print(f"\nProcessing {mouse}")

    selected = load_mouse_last_bouts(mouse, topology)

    if selected is None:
        skipped.append(mouse)
        continue

    summary, bouts, segments = calculate_mouse_metrics(
        mouse, selected, topology
    )

    all_summary.append(summary)
    all_bouts.extend(bouts)
    all_segments.extend(segments)

summary_df = pd.DataFrame(all_summary)
bouts_df = pd.DataFrame(all_bouts)
segments_df = pd.DataFrame(all_segments)

if summary_df.empty:
    raise ValueError("No mice had enough bouts")

summary_df.to_csv(OUTPUT_DIR / "summary.csv", index=False)
bouts_df.to_csv(OUTPUT_DIR / "per_bout.csv", index=False)
segments_df.to_csv(OUTPUT_DIR / "per_segment.csv", index=False)

pd.DataFrame({"mouse_id": skipped}).to_csv(
    OUTPUT_DIR / "skipped_mice.csv",
    index=False
)

settings = {
    "bout_fraction": BOUT_FRACTION,
    "prior_key": PRIOR_KEY,
    "ce_key": CE_KEY,
    "min_segment_transitions": MIN_SEGMENT_TRANSITIONS,
    "k_neighbors": K_NEIGHBORS,
    "min_local_dest_depth": MIN_LOCAL_DEST_DEPTH,
    "min_local_points": MIN_LOCAL_POINTS,
    "mouse_ids": summary_df["mouse_id"].tolist()
}

with open(OUTPUT_DIR / "settings.json", "w") as f:
    json.dump(settings, f, indent=2)

print(f"\nSaved {len(summary_df)} mice")
print(f"Output directory: {OUTPUT_DIR}")


In [ ]:
mice_metrics = pd.read_csv(
    OUTPUT_DIR / "summary.csv",
    dtype={"mouse_id": str}
)

columns = [
    "mouse_id",
    "n_transitions",
    "n_bouts",
    "linear_depth_r2",
    "linear_depth_mae",
    "tortuosity",
    "transition_consistency",
    "transition_balanced_ce",
    "transition_coverage",
    "success_rate",
    "steps_success",
    "node_entropy_normalized",
]

display(
    mice_metrics[columns]
    .round(3)
    .sort_values("mouse_id")
)

## Asymmetric maze

In [ ]:
def build_asymmetric_binary_maze_adjacency() -> np.ndarray:
    """
        Construct the 127-node asymmetric binary tree.
        Actions: -1 (left), 1 (right), 2 (parent).
    """

    num_nodes = 127
    adj = np.zeros((num_nodes, num_nodes), dtype=np.int64)

    # parent: (left_child, right_child)
    children = {
        0: (1, 2),

        1: (3, 4),
        2: (5, 6),

        3: (7, 8),
        4: (9, 10),
        5: (11, 12),
        6: (13, 14),

        7: (15, 16),
        8: (17, 18),
        9: (19, 20),
        10: (21, 22),
        11: (23, 24),
        12: (25, 26),
        13: (27, 28),
        14: (29, 30),

        15: (31, 32),
        16: (33, 34),
        17: (35, 36),
        18: (37, 38),
        19: (39, 40),
        20: (41, 42),
        21: (43, 44),
        22: (45, 46),
        23: (47, 48),
        24: (49, 50),
        26: (51, 52),
        27: (53, 54),
        28: (55, 56),
        29: (57, 58),
        30: (59, 60),

        31: (61, 62),
        32: (63, 64),
        33: (65, 66),
        34: (67, 68),
        36: (69, 70),
        37: (71, 72),
        38: (73, 74),
        40: (75, 76),
        44: (77, 78),
        45: (79, 80),
        46: (81, 82),
        50: (83, 84),
        57: (85, 86),
        58: (87, 88),
        59: (89, 90),
        60: (91, 92),

        62: (93, 94),
        63: (95, 96),
        64: (97, 98),
        66: (99, 100),
        67: (101, 102),
        68: (103, 104),
        71: (105, 106),
        80: (107, 108),
        86: (109, 110),
        91: (111, 112),
        92: (113, 114),
        93: (115, 116),
        95: (117, 118),
        96: (119, 120),
        98: (121, 122),

        111: (123, 124),
        122: (125, 126),
    }

    # Verify that every node except the root has exactly one parent.
    all_children = [
        child
        for child_pair in children.values()
        for child in child_pair
    ]

    assert len(children) == 63
    assert len(all_children) == 126
    assert len(set(all_children)) == 126
    assert set(all_children) == set(range(1, num_nodes))

    for parent, (left_child, right_child) in children.items():
        # Forward actions
        adj[parent, left_child] = -1
        adj[parent, right_child] = 1

        # Return-to-parent action
        adj[left_child, parent] = 2
        adj[right_child, parent] = 2

    return adj

In [ ]:
adj = build_asymmetric_binary_maze_adjacency()
topology = MazeTopology.from_matrix(adj)

assert topology.n_nodes == 127
assert topology.depth.max() == 9
assert topology.depth[126] == 9
assert sum(len(n) for n in topology.neighbors) == 252

ASYM_TOPOLOGY_FILE = Path("asymmetric_topology.npz")
np.savez(ASYM_TOPOLOGY_FILE, adjacency=adj)

print("Saved:", ASYM_TOPOLOGY_FILE)

In [ ]:

DATA_ROOT = Path("runs")
OUTPUT_DIR = Path("maze_metric_outputs_asymmetric")

RUN_SETS = [
    dict(
        name="Asymmetric",
        root=DATA_ROOT / "ASSYM_MAZE_REWARD_DRIVEN_1000",
        prefix="ASSYM_MAZE_REWARD_DRIVEN_1000_runs_slurm_20_window_agent_rew_p_",
        template="{prefix}{p_text}/{seed}",
        ps=[
            0.0, 0.005, 0.01, 0.02,
            0.03, 0.04, 0.05, 0.06,
            0.07, 0.08, 0.09, 0.1
        ],
        seeds=list(range(15)),
        topology_file=ASYM_TOPOLOGY_FILE,
        reward_node=126,
        expected_bouts=1000,
    ),
]

WINDOWS = [
    (0.1, 0.2),
    (0.2, 0.3),
    (0.3, 0.4),
    (0.4, 0.5),
    (0.5, 0.6),
    (0.6, 0.7),
    (0.7, 0.8),
    (0.8, 0.9),
    (0.9, 1.0),
]

OBS_KEY = "obs_full"
ACTION_KEY = "act_full"
PRIOR_KEY = "z_p"
CE_KEY = "ce_p"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
specs = make_run_specs()

print(f"Runs: {len(specs)}")
print(f"Windows: {len(WINDOWS)}")

execute_analysis(specs)